# Clase 09 — Ejercicio intergrador Clínica MediCare

> Notebook de práctica. Ejecutá las celdas a medida que avanzás en el artículo.

La clínica MediCare lleva varios años registrando turnos médicos en un sistema interno. El sistema fue cambiando de versión con el tiempo, y los datos se fueron exportando a una base de datos SQLite sin demasiado control de calidad.

El director médico necesita un informe con algunas preguntas concretas sobre el funcionamiento de la clínica, pero antes de poder responderlas, alguien tiene que hacer la limpieza. Ese trabajo es el de ustedes, no se les dice qué problemas hay en los datos. Parte del ejercicio es encontrarlos.

> El archivo de la base de datos `medicare.db` ya está disponible para que lo uses en este ejercicio.

### 1) Exploración: ¿qué tiene esta base de datos?

Antes de tocar nada, el primer trabajo es entender qué hay adentro. Conectense a `medicare.db` y respondan estas preguntas.

1. ¿Qué tablas tiene la base de datos?
2. ¿Qué columnas y tipos tiene cada tabla?
3. ¿Cuántas filas tiene cada tabla?

In [1]:
import sqlite3
import pandas as pd
from IPython.display import display

with sqlite3.connect("medicare.db") as conexion:
    tablas = pd.read_sql(
        "SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name",
        conexion
    )["name"].tolist()

    for tabla in tablas:
        estructura = pd.read_sql(f"PRAGMA table_info({tabla})", conexion)
        cantidad = pd.read_sql(f"SELECT COUNT(*) AS filas FROM {tabla}", conexion)
        print(f"\n{tabla.upper()}: {cantidad.loc[0, 'filas']} filas")
        display(estructura[["name", "type"]])

# Hallazgo: hay tres tablas: pacientes (66), medicos (20) y turnos (210).


MEDICOS: 20 filas


,name,type
0,id,INTEGER
1,nombre,TEXT
2,especialidad,TEXT
3,matricula,TEXT
4,activo,TEXT



PACIENTES: 66 filas


,name,type
0,id,INTEGER
1,nombre,TEXT
2,dni,TEXT
3,fecha_nacimiento,TEXT
4,ciudad,TEXT
5,obra_social,TEXT



TURNOS: 210 filas


,name,type
0,id,INTEGER
1,paciente_id,INTEGER
2,medico_id,INTEGER
3,fecha,TEXT
4,hora,TEXT
5,motivo,TEXT
6,costo,TEXT
7,estado,TEXT


**Documenten sus hallazgos.** Antes de seguir al Paso 2, escriban en comentarios qué encontraron.

### 2) Diagnóstico: ¿qué problemas tienen los datos?

Traigan las tres tablas a DataFrames y hagan un diagnóstico completo de cada una. Para cada tabla respondan:

- ¿Cuántos nulos hay por columna? ¿Qué porcentaje representan?
- ¿Hay filas duplicadas? ¿Cuáles?
- ¿Hay columnas con tipos incorrectos que van a impedir operar con los datos?
- ¿Hay valores que parecen errores aunque no sean nulos? (Pista: revisen la columna `costo` en `turnos`)


In [2]:
with sqlite3.connect("medicare.db") as conexion:
    df_pacientes = pd.read_sql("SELECT * FROM pacientes", conexion)
    df_medicos = pd.read_sql("SELECT * FROM medicos", conexion)
    df_turnos = pd.read_sql("SELECT * FROM turnos", conexion)

dataframes = {
    "pacientes": df_pacientes,
    "medicos": df_medicos,
    "turnos": df_turnos,
}

for nombre, df in dataframes.items():
    print(f"\n=== {nombre.upper()} ===")
    diagnostico_nulos = pd.DataFrame({
        "nulos": df.isna().sum(),
        "porcentaje": (df.isna().mean() * 100).round(1),
    })
    display(diagnostico_nulos[diagnostico_nulos["nulos"] > 0])
    print(f"Duplicados exactos: {df.duplicated().sum()}")
    print("Tipos:", df.dtypes.astype(str).to_dict())

print("\nPacientes repetidos por DNI (sin contar DNI nulo):",
      df_pacientes.loc[df_pacientes["dni"].notna(), "dni"].duplicated().sum())
print("Médicos repetidos por matrícula (sin contar matrícula nula):",
      df_medicos.loc[df_medicos["matricula"].notna(), "matricula"].duplicated().sum())
print("Turnos duplicados por sus datos de negocio (sin id):",
      df_turnos.duplicated(subset=[c for c in df_turnos.columns if c != "id"]).sum())

costo_numerico = pd.to_numeric(df_turnos["costo"], errors="coerce")
print("Costos no convertibles o vacíos:", costo_numerico.isna().sum())
print("Costos no positivos:", (costo_numerico <= 0).sum())
print("Estados sin normalizar:")
display(df_turnos["estado"].value_counts(dropna=False))


=== PACIENTES ===


,nulos,porcentaje
dni,7,10.6
fecha_nacimiento,11,16.7
ciudad,8,12.1
obra_social,12,18.2


Duplicados exactos: 0
Tipos: {'id': 'int64', 'nombre': 'str', 'dni': 'str', 'fecha_nacimiento': 'str', 'ciudad': 'str', 'obra_social': 'str'}

=== MEDICOS ===


,nulos,porcentaje
matricula,2,10.0
activo,4,20.0


Duplicados exactos: 0
Tipos: {'id': 'int64', 'nombre': 'str', 'especialidad': 'str', 'matricula': 'str', 'activo': 'str'}

=== TURNOS ===


,nulos,porcentaje
hora,16,7.6
motivo,16,7.6
costo,9,4.3


Duplicados exactos: 0
Tipos: {'id': 'int64', 'paciente_id': 'int64', 'medico_id': 'int64', 'fecha': 'str', 'hora': 'str', 'motivo': 'str', 'costo': 'str', 'estado': 'str'}

Pacientes repetidos por DNI (sin contar DNI nulo): 8
Médicos repetidos por matrícula (sin contar matrícula nula): 1
Turnos duplicados por sus datos de negocio (sin id): 1
Costos no convertibles o vacíos: 20
Costos no positivos: 5
Estados sin normalizar:


estado
realizado      105
ausente         29
pendiente       25
cancelado       22
 realizado       6
pendiente        5
AUSENTE          5
realzado         5
Cancelado        3
Realizado        3
 cancelado       2
Name: count, dtype: int64

**Problemas encontrados (antes de limpiar):**

- `pacientes`: nulos en DNI, fecha de nacimiento, ciudad y obra social; 8 DNI repetidos (16 filas involucradas).
- `medicos`: 2 matrículas y 4 estados de actividad nulos; una matrícula repetida.
- `turnos`: nulos en hora, motivo y costo; un turno repetido si se ignora el ID.
- `costo` es texto, contiene `N/A`, cadenas vacías y cinco importes negativos.
- `estado` tiene mayúsculas, espacios y el error tipográfico `realzado`, por lo que no se puede filtrar de forma fiable.

### 3) Decisiones: ¿qué hacemos con cada problema?

Este es el paso más importante del ejercicio. Para cada problema que encontraron en el Paso 2, decidan qué van a hacer y por qué. No hay una respuesta única correcta — lo que importa es que la decisión sea coherente con el análisis que necesitan hacer después.

Completen esta tabla antes de escribir código:

| Tabla | Columna | Problema | Decisión | Justificación |
|-------|---------|----------|----------|---------------|
| pacientes | DNI duplicado | 8 repeticiones | Conservar el primer registro por DNI y remapear sus turnos al ID conservado | El DNI identifica a la persona; así no se pierden turnos por borrar la fila repetida. |
| pacientes | ciudad / obra social | Nulos | Completar como `Sin datos` / `Sin obra social` | Son datos descriptivos; conservar el paciente es más importante. |
| pacientes | DNI / fecha nacimiento | Nulos | Conservar como nulo | No se puede inventar una identidad ni una fecha. |
| medicos | matrícula repetida | 1 repetición | Conservar el primer médico por matrícula y remapear sus turnos | La matrícula es el identificador profesional. |
| medicos | matrícula / activo | Nulos | `Sin datos` en matrícula y booleano nulo en activo | No se debe suponer matrícula ni actividad. |
| turnos | duplicado lógico | 1 repetición | Conservar el primer turno | Repite la misma cita con otro ID. |
| turnos | hora / motivo | Nulos | Completar como `Sin datos` | No impiden contar turnos ni facturar. |
| turnos | costo inválido o negativo | 20 no convertibles, 5 negativos | Convertir a número y dejar como nulo el costo inválido | No se debe facturar un valor desconocido o negativo. |
| turnos | estado | Variantes de escritura | Normalizar y corregir `realzado` a `realizado` | Hace comparables los estados para el análisis. |


### 4) Limpieza: aplicar las decisiones

Ahora sí, implementen las decisiones del Paso 3. Recuerden:

- Primero eliminen duplicados
- Después manejen nulos
- Después conviertan tipos

In [3]:
# Limpieza de pacientes
df_pacientes_limpio = df_pacientes.copy()
# DNI es la identidad disponible: construimos el mapa antes de borrar repetidos.
pacientes_con_dni = df_pacientes_limpio.dropna(subset=["dni"])
mapa_pacientes = (pacientes_con_dni.sort_values("id").groupby("dni")["id"].first())
df_pacientes_limpio = df_pacientes_limpio[
    ~df_pacientes_limpio["dni"].notna() | ~df_pacientes_limpio["dni"].duplicated()
].copy()
df_pacientes_limpio["ciudad"] = df_pacientes_limpio["ciudad"].fillna("Sin datos")
df_pacientes_limpio["obra_social"] = df_pacientes_limpio["obra_social"].fillna("Sin obra social")
df_pacientes_limpio["fecha_nacimiento"] = pd.to_datetime(
    df_pacientes_limpio["fecha_nacimiento"], errors="coerce"
)


# Limpieza de médicos
df_medicos_limpio = df_medicos.copy()
medicos_con_matricula = df_medicos_limpio.dropna(subset=["matricula"])
mapa_medicos = (medicos_con_matricula.sort_values("id").groupby("matricula")["id"].first())
df_medicos_limpio = df_medicos_limpio[
    ~df_medicos_limpio["matricula"].notna() | ~df_medicos_limpio["matricula"].duplicated()
].copy()
df_medicos_limpio["matricula"] = df_medicos_limpio["matricula"].fillna("Sin datos")
df_medicos_limpio["activo"] = df_medicos_limpio["activo"].map({"1": True, "0": False}).astype("boolean")


# Limpieza de turnos
df_turnos_limpio = df_turnos.copy()
# Reasignamos las claves foráneas antes de quitar los registros maestros repetidos.
mapa_id_paciente = df_pacientes.set_index("id")["dni"].map(mapa_pacientes).fillna(df_pacientes.set_index("id").index.to_series())
mapa_id_medico = df_medicos.set_index("id")["matricula"].map(mapa_medicos).fillna(df_medicos.set_index("id").index.to_series())
df_turnos_limpio["paciente_id"] = df_turnos_limpio["paciente_id"].map(mapa_id_paciente).astype(int)
df_turnos_limpio["medico_id"] = df_turnos_limpio["medico_id"].map(mapa_id_medico).astype(int)
df_turnos_limpio = df_turnos_limpio.drop_duplicates(
    subset=[c for c in df_turnos_limpio.columns if c != "id"], keep="first"
).copy()
df_turnos_limpio["hora"] = df_turnos_limpio["hora"].fillna("Sin datos")
df_turnos_limpio["motivo"] = df_turnos_limpio["motivo"].fillna("Sin datos")
df_turnos_limpio["costo"] = pd.to_numeric(df_turnos_limpio["costo"], errors="coerce")
df_turnos_limpio.loc[df_turnos_limpio["costo"] <= 0, "costo"] = pd.NA
df_turnos_limpio["estado"] = (
    df_turnos_limpio["estado"].str.strip().str.lower().replace({"realzado": "realizado"})
)
df_turnos_limpio["fecha"] = pd.to_datetime(df_turnos_limpio["fecha"], errors="coerce")

> **Por qué `.copy()`:** cuando hacen `df_pacientes_limpio = df_pacientes`, no están creando una copia — están apuntando al mismo objeto. Cualquier cambio en `df_pacientes_limpio` modifica también `df_pacientes`. `.copy()` crea una copia independiente, así conservan los datos originales para comparar.

Al terminar, impriman un resumen de qué cambió en cada tabla:

In [4]:
for nombre, original, limpio in [
    ("Pacientes", df_pacientes, df_pacientes_limpio),
    ("Médicos", df_medicos, df_medicos_limpio),
    ("Turnos", df_turnos, df_turnos_limpio),
]:
    print(f"\n{nombre}: {len(original)} → {len(limpio)} filas")
    print("Nulos finales:", limpio.isna().sum().to_dict())
    print("Tipos principales:", limpio.dtypes.astype(str).to_dict())


Pacientes: 66 → 58 filas
Nulos finales: {'id': 0, 'nombre': 0, 'dni': 7, 'fecha_nacimiento': 10, 'ciudad': 0, 'obra_social': 0}
Tipos principales: {'id': 'int64', 'nombre': 'str', 'dni': 'str', 'fecha_nacimiento': 'datetime64[us]', 'ciudad': 'str', 'obra_social': 'str'}

Médicos: 20 → 19 filas
Nulos finales: {'id': 0, 'nombre': 0, 'especialidad': 0, 'matricula': 0, 'activo': 4}
Tipos principales: {'id': 'int64', 'nombre': 'str', 'especialidad': 'str', 'matricula': 'str', 'activo': 'boolean'}

Turnos: 210 → 209 filas
Nulos finales: {'id': 0, 'paciente_id': 0, 'medico_id': 0, 'fecha': 0, 'hora': 0, 'motivo': 0, 'costo': 25, 'estado': 0}
Tipos principales: {'id': 'int64', 'paciente_id': 'int64', 'medico_id': 'int64', 'fecha': 'datetime64[us]', 'hora': 'str', 'motivo': 'str', 'costo': 'float64', 'estado': 'str'}


### 5) — Análisis: responder las preguntas del director

Con los datos limpios, respondan estas cuatro preguntas. Para cada una, combinen SQL (para traer los datos necesarios) con Pandas (para calcular o presentar el resultado):

**Pregunta 1:** ¿Cuántos turnos realizó cada médico? Mostrá el resultado ordenado de mayor a menor.

**Pregunta 2:** ¿Cuál es el costo total facturado por especialidad médica? Considerá solo los turnos con estado `'realizado'`.

**Pregunta 3:** ¿Qué obra social tiene más pacientes registrados? ¿Y cuántos pacientes no tienen obra social?

**Pregunta 4:** ¿Cuántos turnos fueron cancelados o marcados como `'ausente'`? ¿Qué médico tuvo más cancelaciones?


### 6) Reporte final

Escriban un bloque de código que imprima un reporte con los resultados de los cuatro puntos anteriores, en formato legible. No es necesario que sea elaborado — lo importante es que quede claro qué encontraron.

In [5]:
# Las tablas ya fueron extraídas con SQL; aquí Pandas combina las versiones limpias.
turnos_con_medicos = df_turnos_limpio.merge(
    df_medicos_limpio[["id", "nombre", "especialidad"]],
    left_on="medico_id", right_on="id", how="left", validate="many_to_one"
)

turnos_por_medico = (
    turnos_con_medicos.groupby("nombre").size().sort_values(ascending=False)
)
facturacion_por_especialidad = (
    turnos_con_medicos.loc[turnos_con_medicos["estado"] == "realizado"]
    .groupby("especialidad")["costo"].sum(min_count=1)
    .sort_values(ascending=False)
)
obras_sociales = df_pacientes_limpio["obra_social"].value_counts()
obra_principal = obras_sociales.index[0]
pacientes_sin_obra = (df_pacientes_limpio["obra_social"] == "Sin obra social").sum()

turnos_cancelados = (df_turnos_limpio["estado"] == "cancelado").sum()
turnos_ausentes = (df_turnos_limpio["estado"] == "ausente").sum()
cancelaciones_por_medico = (
    turnos_con_medicos.loc[turnos_con_medicos["estado"] == "cancelado"]
    .groupby("nombre").size().sort_values(ascending=False)
)

print("=" * 50)
print("REPORTE CLÍNICA MEDICARE")
print("=" * 50)

print("\n1. Turnos por médico:")
print(turnos_por_medico.to_string())

print("\n2. Facturación por especialidad (solo realizados y con costo válido):")
for especialidad, total in facturacion_por_especialidad.items():
    print(f"- {especialidad}: ${total:,.0f}")

print("\n3. Obra social con más pacientes:")
print(f"- {obra_principal}: {obras_sociales.iloc[0]} pacientes")
print(f"- Sin obra social: {pacientes_sin_obra} pacientes")

print("\n4. Estados de inasistencia:")
print(f"- Cancelados: {turnos_cancelados}")
print(f"- Ausentes: {turnos_ausentes}")
print(f"- Total cancelados o ausentes: {turnos_cancelados + turnos_ausentes}")
print(f"- Más cancelaciones: {cancelaciones_por_medico.index[0]} ({cancelaciones_por_medico.iloc[0]})")

print("\nNota: los costos inválidos y negativos se excluyeron de facturación; no se inventaron importes.")

REPORTE CLÍNICA MEDICARE

1. Turnos por médico:
nombre
Dra. Beatriz Sosa      33
Dr. Sebastián Ríos     31
Dra. Carmen Aguirre    28
Dr. Nicolás Vera       18
Dr. Pablo Méndez       18
Dra. Silvina Vega      13
Dra. Laura Giménez     13
Dra. Gabriela Luna     11
Dra. Andrea Blanco     11
Dr. Ezequiel Bravo     10
Dr. Martín Cufré        9
Dr. Hernán Costa        9
Dra. Verónica Paz       5

2. Facturación por especialidad (solo realizados y con costo válido):
- Traumatología: $120,000
- Dermatología: $108,000
- Cardiología: $100,800
- Pediatría: $94,300
- Clínica Médica: $84,000
- Ginecología: $44,300
- Psiquiatría: $41,500
- Endocrinología: $24,700
- Neurología: $11,000

3. Obra social con más pacientes:
- Swiss Medical: 12 pacientes
- Sin obra social: 12 pacientes

4. Estados de inasistencia:
- Cancelados: 27
- Ausentes: 34
- Total cancelados o ausentes: 61
- Más cancelaciones: Dra. Beatriz Sosa (4)

Nota: los costos inválidos y negativos se excluyeron de facturación; no se inventaro

→ [Ir a la solución](./solucion.md)